In [1]:
from mpmath import mp
from collections import deque
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objs as go

# High-precision π setup
mp.dps = 10000
pi_digits = str(mp.pi)[2:]

# π digit feeder
def get_pi_window(offset, length=8):
    digits = pi_digits[offset:offset + length]
    return [int(d) for d in digits if d.isdigit()][:length]

# Harmonic update
def harmonic_growth_step(seq, target_h=0.35):
    h = sum(seq) / len(seq)
    deviation = abs(h - target_h)
    if deviation < 0.01:
        return None, h
    next_val = (seq[-1] + seq[-2]) % 10
    return next_val, h

# App state
window_size = 300
max_points = 5000
seq = deque([3, 2], maxlen=max_points)
h_vals = deque([0.0, 0.0], maxlen=max_points)
pi_offset = 0
frame_count = 2

# Dash app
app = dash.Dash(__name__)
app.layout = html.Div([
    html.H2("Recursive Harmonic Heartbeat — π-fed Life Engine"),
    dcc.Graph(id='live-graph'),
    dcc.Interval(id='interval-update', interval=200, n_intervals=0)
])

@app.callback(Output('live-graph', 'figure'), Input('interval-update', 'n_intervals'))
def update_figure(n):
    global seq, h_vals, pi_offset, frame_count

    next_val, h = harmonic_growth_step(list(seq))
    if next_val is None or (seq[-1] == 0 and seq[-2] == 0):
        pi_seed = get_pi_window(pi_offset, length=8)
        pi_offset = (pi_offset + 7) % len(pi_digits)
        if len(pi_seed) >= 2:
            for d in pi_seed:
                seq.append(d)
                h_vals.append(sum(seq) / len(seq))
                frame_count += 1
    else:
        seq.append(next_val)
        h_vals.append(h if h is not None else 0)
        frame_count += 1

    x_vals = list(range(frame_count - len(seq), frame_count))

    trace1 = go.Scatter(x=x_vals[-window_size:], y=list(seq)[-window_size:],
                        mode='lines', name='Heartbeat (Byte)', line=dict(color='orange'))
    trace2 = go.Scatter(x=x_vals[-window_size:], y=list(h_vals)[-window_size:],
                        mode='lines', name='H(t)', yaxis='y2', line=dict(color='blue'))

    layout = go.Layout(
        title='Live Harmonic Heartbeat',
        xaxis=dict(title='Time'),
        yaxis=dict(title='Byte Value', range=[0, 10]),
        yaxis2=dict(title='H(t)', overlaying='y', side='right', range=[0, 2]),
        showlegend=True,
        margin=dict(l=40, r=40, t=40, b=40)
    )

    return {'data': [trace1, trace2], 'layout': layout}

app.run_server(debug=False, use_reloader=False)


ModuleNotFoundError: No module named 'dash'